In [ ]:
import os
from groq import Groq

# Paste the NEW key here — copy it directly from console.groq.com, no extra spaces
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY".strip()

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Quick sanity check before your real query
test = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Say hello in one word."}]
)
print(test.choices[0].message.content)

Hello


In [42]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from groq import Groq
from google.colab import files

# 1. Upload orders.xlsx (skip if already uploaded)
uploaded = files.upload()

# 2. Load and prep data
df = pd.read_excel("orders.xlsx")

def row_to_text(row):
    return (f"Order {row['Order ID']} placed by {row['Customer Name']} "
            f"for {row['Product']}, shipped to {row['Shipping Address']} "
            f"on {row['Order Date']}. Status: {row['Status']}.")

df["text"] = df.apply(row_to_text, axis=1)

# 3. Build search index
vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(df["text"])

def search(query, k=5):
    q_vec = vectorizer.transform([query])
    scores = (doc_vectors @ q_vec.T).toarray().flatten()
    top_idx = scores.argsort()[-k:][::-1]
    return df.iloc[top_idx]

# 4. Ask a question
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

query = "which orders are cancelled?"   # change this to test other questions
results = search(query, k=5)
context = "\n".join(results["text"].tolist())

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": "Answer strictly based on the order data provided. If the answer isn't in the data, say so."},
        {"role": "user", "content": f"Order data:\n{context}\n\nQuestion: {query}"}
    ]
)

print("Matched rows:\n", results[["Order ID", "Customer Name", "Status"]])
print("\nGroq's Answer:\n", response.choices[0].message.content)

Saving orders.xlsx to orders (7).xlsx
Matched rows:
     Order ID  Customer Name     Status
46  ORD-1047   Andrew Malik  Cancelled
14  ORD-1015  Ayesha Rashid  Cancelled
22  ORD-1023    George Aziz  Cancelled
32  ORD-1033     Jane Malik  Cancelled
7   ORD-1008   Noor Johnson  Cancelled

Groq's Answer:
 The following orders have a status of **Cancelled**:

- ORD-1047  
- ORD-1015  
- ORD-1023  
- ORD-1033  
- ORD-1008


In [43]:
test_queries = [
    "list all orders shipped to Lahore",
    "who ordered a Bluetooth Speaker?",
    "how many orders are pending?",
    "show me delivered orders in Karachi",
    "which customer has the most orders?"
]

for q in test_queries:
    results = search(q, k=5)
    context = "\n".join(results["text"].tolist())

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": "Answer strictly based on the order data provided. If the answer isn't in the data, say so."},
            {"role": "user", "content": f"Order data:\n{context}\n\nQuestion: {q}"}
        ]
    )

    print(f"Q: {q}")
    print(f"A: {response.choices[0].message.content}")
    print("-" * 60)

Q: list all orders shipped to Lahore
A: Orders shipped to Lahore:

- **ORD-1044** – Layla Sheikh – Smartwatch – Delivered (shipped to 313 Mall Rd, Lahore, Punjab 54000)  
- **ORD-1009** – William Shah – Smartwatch – Processing (shipped to 731 Park Ave, Lahore, Punjab 54000)
------------------------------------------------------------
Q: who ordered a Bluetooth Speaker?
A: Mariam Johnson ordered a Bluetooth Speaker (Order ORD-1010).
------------------------------------------------------------
Q: how many orders are pending?
A: There are **4 pending orders**.
------------------------------------------------------------
Q: show me delivered orders in Karachi
A: **Delivered orders shipped to Karachi**

| Order ID | Customer | Product | Shipping Address | Delivery Date |
|----------|----------|---------|------------------|---------------|
| ORD-1006 | Adam Nazir | Portable Charger | 104 Jail Rd, Karachi, Sindh 74200 | 2026‑08‑23 |
| ORD-1027 | Ava Ahmed | Screen Protector | 909 College Rd, 